# 08 · 응력 해석 — 비균열·균열·사용·극한

원 문서의 `stress_analysis.ipynb` 에 대응한다.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [3]:
kds, conc_sec = beam_section()

cracked = kds.calculate_cracked_properties(theta=0)
m_service = 1.5 * cracked.m_cr
_, u_res, _ = kds.ultimate_bending_capacity()

print(f"균열모멘트   Mcr = {cracked.m_cr / 1e6:.2f} kN.m")
print(f"사용 모멘트  M   = {m_service / 1e6:.2f} kN.m  (= 1.5 Mcr)")

균열모멘트   Mcr = 88.42 kN.m
사용 모멘트  M   = 132.63 kN.m  (= 1.5 Mcr)


In [4]:
uncracked = kds.calculate_uncracked_stress(m_x=m_service)
cracked_stress = kds.calculate_cracked_stress(
    cracked_results=cracked, m=m_service
)
service = kds.calculate_service_stress(
    moment_curvature_results=kds.moment_curvature_analysis(
        theta=0, kappa_inc=1e-7, progress_bar=False
    ),
    m=m_service,
)
ultimate = kds.calculate_ultimate_stress(ultimate_results=u_res)


def summarise(label, res):
    conc_s = [float(s) for arr in res.concrete_stresses for s in arr]
    steel_s = [float(s) for s in res.lumped_reinforcement_stresses]
    print(
        f"{label:>8} | concrete {min(conc_s):8.2f} ~ {max(conc_s):7.2f} MPa"
        f" | steel {min(steel_s):9.2f} ~ {max(steel_s):8.2f} MPa"
    )


summarise("비균열", uncracked)
summarise("균열", cracked_stress)
summarise("사용", service)
summarise("극한", ultimate)

     비균열 | concrete    -4.91 ~    5.15 MPa | steel    -29.12 ~    31.26 MPa
      균열 | concrete     0.00 ~    8.75 MPa | steel   -174.93 ~    39.68 MPa
      사용 | concrete     0.00 ~    8.75 MPa | steel   -174.96 ~    39.69 MPa
      극한 | concrete     0.00 ~   22.95 MPa | steel   -400.00 ~   160.29 MPa


극한 상태의 콘크리트 압축응력 22.95 MPa 는 $\eta(0.85f_{ck})$ 와,
철근 인장응력 −400 MPa 는 SD400 의 항복강도와 일치한다.

In [5]:
for label, res in [
    ("Uncracked", uncracked),
    ("Cracked", cracked_stress),
    ("Service", service),
    ("Ultimate", ultimate),
]:
    res.plot_stress(title=label)

비균열 해석은 인장측 콘크리트가 응력을 받는 것으로 보고, 균열 해석은
인장을 무시한다. 사용 해석은 모멘트-곡률 결과를 이용해 실제 응력-변형률
관계를 따른다.